# Prep

## Import stuff

In [1]:
from pathlib import Path
import pandas as pd
pd.set_option('max_colwidth', 100)
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix
import itertools
import scipy.stats as st
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
#import seaborn as sns
#from matplotlib import pyplot as plt
#%matplotlib inline
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 500)
import rpy2
#import pingouin as pg
from itertools import combinations
import openpyxl
from contextlib import redirect_stdout
import random
import math

## Some magical magic to make the R stuff work

In [2]:
import os
os.environ["OMP_NUM_THREADS"], os.environ["OPENBLAS_NUM_THREADS"], os.environ["MKL_NUM_THREADS"] 

('1', '1', '1')

In [3]:
# The rpy2/R session and CFA machinery now live in the hitop_cfa package.
# Importing it starts the embedded R session: it sets the BLAS threading
# env vars (if unset) and loads base, utils, lavaan, and the patched semTools.
import hitop_cfa
from hitop_cfa.r_env import (ro, rbase, utils, lavaan, semtools,
                             RRuntimeError, pandas2ri, localconverter)

import rpy2.ipython.html
rpy2.ipython.html.init_printing()

# Paths

In [4]:
# paths where to save preprocessed data files
log_dir = Path('./log')
log_dir.mkdir(exist_ok=True)
dat_dir = Path('../data/')
val_dir = dat_dir / 'ValSample'
fin_dir = dat_dir / 'finaldata'
cfa_dir = dat_dir / 'cfa'
cfa_dir.mkdir(exist_ok=True)
path_save_val = fin_dir / 'dat_val.csv'
path_save_dat_gp_grid1st_norecontact = fin_dir / 'dat_gp_grid1st_norecontact.csv'
path_save_dat_en_grid1st_norecontact = fin_dir / 'dat_en_grid1st_norecontact.csv'
path_save_dat_gp_grid1st_full = fin_dir / 'dat_gp_grid1st_full.csv'
path_save_dat_en_grid1st_full = fin_dir / 'dat_en_grid1st_full.csv'
path_save_dat_gp_gridall_full = fin_dir / 'dat_gp_gridall_full.csv'
path_save_dat_en_gridall_full = fin_dir / 'dat_en_gridall_full.csv'
# path_save_dat_gp_gridall_recontact = '../../data/finaldata/dat_gp_gridall_recontact.csv'
# path_save_dat_en_gridall_recontact = '../../data/finaldata/dat_en_gridall_recontact.csv'
# helped file for cfa
helpfile_dir = cfa_dir / 'temp'
helpfile_dir.mkdir(exist_ok=True, parents=True)
path_to_helpfile = helpfile_dir / 'cfa_temp.csv'
path_to_cogmood_questions = dat_dir / 'cogmood_questions.csv'
path_to_item_lookup = val_dir / 'Internalizing-Somatoform Items_DW.xlsx'

## Count how many cpus I have, then decide how many I want to use; define how many iterations for CFA (decrease for debugging)

In [5]:
total_cpus = os.cpu_count()
# account for hyperthreading
cpus_to_use = total_cpus//2 - 1
global cpus_to_use
print(f"\nGoing to use {cpus_to_use} CPUs for CFA heavy-lifting\n")

num_iter = 1000
global num_iter


Going to use 5.0 CPUs for CFA heavy-lifting



## SET SEEDS !!!!!!!!!!

In [6]:
#rngkind = "L'Ecuyer-CMRG"
random.seed(12345)

In [7]:
ro.r('RNGkind(kind = "L\'Ecuyer-CMRG")')
ro.r('set.seed(12345)')

<rpy2.rinterface_lib.sexp.NULLType object at 0x126b11710> [0]

### TEST THE SEEDS!!!!!!!!

In [8]:
for i in range(5):
    print(random.random())
# after kernel restart, this should be 
# 0.41661987254534116
# 0.010169169457068361
# 0.8252065092537432
# 0.2986398551995928
# 0.3684116894884757

0.41661987254534116
0.010169169457068361
0.8252065092537432
0.2986398551995928
0.3684116894884757


In [9]:
ro.r('rnorm(5)')
# after kernel restart, this should be 
# -1.457850350316457	-0.45246126454182867	0.3650586371545244	-1.57091128601566	1.1419085835874878

-1.457850350316457,-0.45246126454182867,0.36505863715452436,-1.57091128601566,1.1419085835874876


### I'M ALSO SETTING THE SAME SEEDS EVERY TIME I RUN THE HELPED CFA FUNCTION, JUST IN CASE!!!!!

# Functions

## CFA helper functions

In [10]:
from hitop_cfa import (
    build_luts,
    check_hitop_ids,
    cfa_helper_func,
    run_specific_cfa,
    do_three_way_cfa_stepwise_mi,
    do_three_way_cfa_stepwise_scalar,
    exhaustive_cfa_ablations,
    set_seeds,
    silence_r,
)

# item-text lookups (was load_item_lookup + inline lut construction)
_luts = build_luts(path_to_item_lookup, path_to_cogmood_questions)
item_lookup = _luts['item_lookup']
item_lut = _luts['item_lut']
phq_lut = _luts['phq_lut']
gad_lut = _luts['gad_lut']
baars_lut = _luts['baars_lut']

/Users/nielsond/code/hitop_val/HiTOP/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/HiTOP/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/HiTOP/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/HiTOP/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


## CFA wrapper functions

# Run Main Code

## Load preprocessed data and concatinate

In [11]:
# load
data_val = pd.read_csv(path_save_val)

data_gp = pd.read_csv(path_save_dat_gp_grid1st_full)
data_en = pd.read_csv(path_save_dat_en_grid1st_full)

# concatinate
data_val_genpop = pd.concat([data_val, data_gp])
data_val_enriched = pd.concat([data_val, data_en])
data_genpop_enriched = pd.concat([data_gp, data_en])

data_gp_nore = pd.read_csv(path_save_dat_gp_grid1st_norecontact)
data_en_nore = pd.read_csv(path_save_dat_en_grid1st_norecontact)
data_genpop_enriched_norecontact = pd.concat([data_gp_nore, data_en_nore])

# Dataset pairs passed to the hitop_cfa functions. The key strings become
# labels downstream, so they must match what the original code used:
#   run_specific_cfa -> 'pair' column values consumed by Invariance_plot
#   stepwise search  -> per-pair history column names
datasets_runspecific = {'V_GP': data_val_genpop, 'V_EN': data_val_enriched, 'GP_EN': data_genpop_enriched}
datasets_stepwise = {'val_gp': data_val_genpop, 'val_en': data_val_enriched, 'gp_en': data_genpop_enriched}
datasets_gp_en = {'gp_en': data_genpop_enriched}

# Find invariant subsets via stepwise elimination

In [12]:
orig_items = {
    'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop77 + hitop84 + hitop92 + hitop93 + hitop123 + hitop157 + hitop182 + hitop230 + hitop246',
    'anxious_worry': 'anxious_worry =~hitop20 + hitop34 + hitop89 + hitop203 + hitop240 + hitop248 + hitop265',
    'appetite_gain': 'appetite_gain =~hitop120 + hitop141 + hitop243 + hitop275',
    'appetite_loss': 'appetite_loss =~hitop280 + hitop283 + hitop109',
    'cognitive_problems': 'cognitive_problems =~hitop67 + hitop159 + hitop189 + hitop142',
    'hyposomnia': 'hyposomnia =~hitop99 + hitop181 + hitop5 + hitop66 + hitop231',
    'indecisiveness': 'indecisiveness =~hitop21 + hitop90 + hitop95',
    'insomnia': 'insomnia =~hitop160 + hitop254 + hitop261 + hitop268',
    'panic': 'panic =~hitop15 + hitop104 + hitop126 + hitop211 + hitop215 + hitop257',
    'separation_insecurity': 'separation_insecurity =~hitop40 + hitop50 + hitop69 + hitop81 + hitop113 + hitop136 + hitop151 + hitop197',
    'shame_guilt': 'shame_guilt =~hitop72 + hitop140 + hitop143 + hitop220',
    'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247 + hitop278',
    'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop117 + hitop124 + hitop129 + hitop204 + hitop222 + hitop236 + hitop258',
    'well_being': 'well_being =~hitop9 + hitop23 + hitop54 + hitop106 + hitop149 + hitop200 + hitop244 + hitop245 + hitop250 + hitop281'
}
            

In [13]:
with open("log/mylog_3wayCFA_origscales_seed12345.txt", "w") as f:
    with redirect_stdout(f):
        orig_icc_res = []
        for scale, items in orig_items.items():
            # print which scale we are processing through R - this way it doesn't get saved in the log file
            ro.globalenv['scale_to_print'] = scale
            ro.r('print(scale_to_print)')
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test
            icc_res = run_specific_cfa(
                whichscale=scale,
                item_list=items_list,
                whichcfa='strict',
                datasets=datasets_runspecific,
                temp_path=path_to_helpfile,
                num_iter=num_iter,
                cpus_to_use=cpus_to_use,
                return_vals=True
            )
            icc_res = pd.DataFrame(icc_res)
            icc_res['scale'] = scale
            orig_icc_res.append(icc_res)
orig_icc_res = pd.concat(orig_icc_res)

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: 

R[write to console]: Warning messages:

R[write to console]: 1: lavaan->lavTestScore():  
   se is not `standard'; not implemented yet; falling back to ordinary score 
   test 

R[write to console]: 2: lavaan->lavTestScore():  
   se is not `standard'; not implemented yet; falling back to ordinary score 
   test 

R[write to console]: 3: lavaan->lavTestScore():  
   se is not `standard'

RRuntimeError: Error in loadNamespace("emmeans") : there is no package called ‘emmeans’


In [ ]:
orig_icc_res = orig_icc_res.astype(float, errors='ignore')
orig_icc_res = orig_icc_res.replace("NA", pd.NA)

In [ ]:
orig_icc_res.to_csv(log_dir / 'orig_icc_res.csv', index=None)

for a set of items:  
    if it's 3-way config and 3-way metric:  
        return set of items and list of removed items  
    if it's not 3-way config invariant:  
        evaluate configural invariance on all single item ablations
        pick the 3-way config invariant ablation that has the best CFI (with TLI tibreaker if the dif in CFI is less than 0.001 across all 3 pairs  
        restart the loop with the selected set of items  
    if it's 3-way config invariant, but not 3-way metric:  
       remove the item with the highest modification index
       restart the loop with the selected set of items

In [ ]:
# silence R to clean up messages
# be careful doing this, you might miss important warnings
silence_r()

In [ ]:
stepwise_res = []
histories = []
scales_failing_metric = [
    'anhedonic_depression',
    'anxious_worry',
    'appetite_gain',
    'hyposomnia',
    'panic',
    'separation_insecurity',
    'situational_phobia',
    'shame_guilt',
    'social_anxiety',
    'well_being',
]
for scale in scales_failing_metric:
    final, removed, history = do_three_way_cfa_stepwise_mi(
        scale,
        orig_items=orig_items,
        datasets=datasets_stepwise,
        temp_path=path_to_helpfile,
        num_iter=num_iter,
        cpus_to_use=cpus_to_use,
        min_items=3,
    )
    if final is not None:
        final_items = [item_lut[item_no] for item_no in final]
        removed_items = [item_lut[item_no] for item_no in removed]
        row = dict(
            scale=scale,
            item_nos=final,
            removed_nos=removed,
            items=final_items,
            removed=removed_items
        )
    else:
        row = dict(
            scale=scale
        )

    stepwise_res.append(row)
    pd.DataFrame(stepwise_res).to_pickle(cfa_dir / 'stepwise_in_progress.pkl')
    history['scale'] = scale
    history.to_pickle(cfa_dir / f'{scale}_history.pkl')
    histories.append(history)

In [29]:
stepwise_res = pd.DataFrame(stepwise_res)

In [30]:
stepwise_res

,scale,item_nos,removed_nos,items,removed
0,anhedonic_depression,"[hitop77, hitop84, hitop93, hitop123, hitop182, hitop230, hitop246]","[hitop39, hitop157, hitop92]","[I didn’t look forward to seeing friends or family., I felt depressed., Nothing seemed interesti...","[It felt like there wasn’t anything interesting or fun to do., I had very little energy., It too..."
1,anxious_worry,NaN,NaN,NaN,NaN
2,appetite_gain,"[hitop120, hitop243, hitop275]",[hitop141],"[I could not keep myself from eating., I stuffed myself with food., I ate even when I was not re...",[I thought a lot about food.]
3,hyposomnia,"[hitop99, hitop5, hitop66, hitop231]",[hitop181],"[I needed much less sleep than usual., I had days when I never got tired., I did not feel tired,...",[I felt like I could keep going and going without ever getting tired.]
4,panic,"[hitop15, hitop104, hitop126, hitop215, hitop257]",[hitop211],"[I was short of breath., I felt nauseated., My heart was racing or pounding., My hands were cold...",[I was trembling or shaking.]
5,separation_insecurity,"[hitop40, hitop69, hitop81, hitop113, hitop136, hitop151]","[hitop50, hitop197]","[I felt insecure about important relationships in my life., I wanted other people to take care o...","[I wanted someone else to make decisions for me., I could not stand being alone.]"
6,situational_phobia,"[hitop16, hitop165, hitop278]","[hitop225, hitop247]","[I avoided riding in elevators., I was afraid of flying., I became very anxious during a storm.]","[I was afraid of the dark., I was afraid of heights.]"
7,shame_guilt,"[hitop72, hitop140, hitop220]",[hitop143],"[I was disgusted with myself., I blamed myself for things., I felt ashamed of things I had done.]",[I felt guilty.]
8,social_anxiety,"[hitop124, hitop222, hitop258]","[hitop1, hitop117, hitop204, hitop236, hitop129, hitop114, hitop17]","[I avoided situations in which others were likely to watch me., I was uncomfortable entering a r...","[I felt shy around other people., I felt socially awkward., I felt uncomfortable being the cente..."
9,well_being,"[hitop9, hitop23, hitop149, hitop200, hitop244, hitop250, hitop281]","[hitop106, hitop54, hitop245]","[I felt like I was having a lot of fun., I felt cheerful., I felt good about myself., I found co...","[I was proud of myself., It was easy for me to laugh., I looked forward to things with enjoyment.]"


In [31]:
stepwise_res.to_pickle(cfa_dir / 'stepwise.pkl')

# Exploratory: check metric invariance of PHQ, GAD, and BAARS

In [ ]:
other_scales = {
    'phq_sum': 'phq_sum=~phq_1 + phq_2 + phq_3 + phq_4 + phq_5 + phq_6 + phq_7 + phq_8',
    'gad_sum': 'gad_sum =~gad_1 + gad_2 + gad_3 + gad_4 + gad_5 + gad_6 + gad_7',
    'baars_inattention_sum': 'baars_inattention_sum =~inattention_1 + inattention_2 + inattention_3 + inattention_4 + inattention_5 + inattention_6 + inattention_7 + inattention_8 + inattention_9',
    'baars_hyperactivity_sum': 'baars_hyperactivity_sum =~hyperactivity_1 + hyperactivity_2 + hyperactivity_3 + hyperactivity_4 + hyperactivity_5',
    'baars_impulsivity_sum': 'baars_impulsivity_sum =~impulsivity_1 + impulsivity_2 + impulsivity_3 + impulsivity_4',
    'baars_sct_sum': 'baars_sct_sum =~sct_1 + sct_2 + sct_3 + sct_4 + sct_5 + sct_6 + sct_7 + sct_8 + sct_9'}

other_log = log_dir /'mylog_3wayCFA_baarsgadphq_seed12345.txt'
with other_log.open("w") as f:
    with redirect_stdout(f):
        for scale, items in other_scales.items():
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            print(f"Items: {items}")

            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")

            # +++ GENPOP VS ENRICHED +++
            print('\n -----> GENPOP VS ENRICHED <----- ')
            flag_metric_gp_en, pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en = cfa_helper_func(
                            scalename=scale,
                            list_of_items=items_list,
                            do_metric=True,
                            do_scalar=True,
                            do_strict=True,
                            mydata_python=data_genpop_enriched,
                            mydata_temp_path=path_to_helpfile,
                            num_iter=num_iter,
                            cpus_to_use=cpus_to_use)

In [ ]:
other_exhaustive_logs = log_dir /'mylog_exhuastive_inv_search_baarsgadphq_seed12345.txt'

with other_exhaustive_logs.open("w") as f:
    with redirect_stdout(f):
        for scale in ['phq_sum', 'gad_sum', 'baars_inattention_sum', 'baars_sct_sum']:
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            items = other_scales[scale]
            # create a neat FULL list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # how many items are there in total?
            howmany = len(items_list)
            # remove items one by one
            for i in range (1, howmany):
                how_many_to_try = howmany - i
                print(f'\n-----------------------------\nTESTING n - {i} = {how_many_to_try} ITEMS for scale {scale}\n-----------------------------\n')
                if how_many_to_try <= 2: # the min amount of items we can test is 3
                    print("\nWe ran out of items! No inv subset can be found")
                    break
                # test cfa
                successful_combinations_for_scale = exhaustive_cfa_ablations(
                    whichscale=scale,
                    whichcfa='metric',
                    howmanyitems=how_many_to_try,
                    orig_items=other_scales,
                    datasets=datasets_gp_en,
                    temp_path=path_to_helpfile,
                    num_iter=num_iter,
                    cpus_to_use=cpus_to_use,
                )
                # if found any number of successful items, save them and stop trying for this scale
                if successful_combinations_for_scale:
                    print(f'\n!!!!! Found at least one invariant subset for SCALE {scale} with ITEMS = {how_many_to_try} (removing {i} items)\n')
                    print(successful_combinations_for_scale)
                    break

In [36]:
import ast

In [37]:
lfi_out = other_exhaustive_logs.read_text().split('\n')
lfi_parsing = []
for lix, line in enumerate(lfi_out):
    if line.startswith('!!!!!'):
        row= dict(
            lix=lix,
            scale=line.split('SCALE')[-1].split('with')[0].strip(),
            n_items=int(line.split('ITEMS = ')[-1].split(' (')[0]),
            n_removed=int(line.split('removing ')[-1].split(' items')[0])
        )
        lfi_parsing.append(row)
ds_pairs = ['gp_en']
inv_levels = ['config', 'metric', 'scalar', 'strict']

In [38]:
inv_dat = []
for lprow, olut  in zip(lfi_parsing, [phq_lut, gad_lut, baars_lut['inattention'], baars_lut['sct']]):
    scale_stats = ast.literal_eval(lfi_out[lprow['lix'] + 2])
    for issix, iss in enumerate(scale_stats.items()):
        row = lprow.copy()
        item_nos = iss[0]
        ogitems = list(olut.keys())
        items = [olut[item_no] for item_no in item_nos]
        removed_nos = [ii for ii in ogitems if ii not in item_nos]
        removed_items = [olut[item_no] for item_no in removed_nos]
        row['ssix'] = issix
        row['item_nos'] = item_nos
        row['removed_nos'] = removed_nos
        row['items'] = items
        row['removed'] = removed_items
        inv_stats = iss[1]
        for dsp in ds_pairs:
            for lix, level in enumerate(inv_levels):
                p = inv_stats[dsp][lix]
                if p == 'NA':
                    p = np.nan
                else:
                    p = float(p)
                row[f'{dsp}__{level}'] = p
        inv_dat.append(row)
inv_dat = pd.DataFrame(inv_dat)

In [39]:
for row in inv_dat.itertuples():
    print("########################")
    print(f'Scale: {row.scale}, Subset_id: {row.ssix}')
    print("########################")
    for ii in row.items:
        print(ii)
    print('---------REMOVED---------------')
    for ii in row.removed:
        print(ii)
    print()
    print()

########################
Scale: phq_sum, Subset_id: 0
########################
Little interest or pleasure in doing things
Trouble falling or staying asleep, or sleeping too much
Feeling tired or having little energy
Poor appetite or overeating
Feeling bad about yourself – or that you are a failure or have let yourself or your family down
Trouble concentrating on things, such as school work, reading or watching television
---------REMOVED---------------
Feeling down, depressed, irritable or hopeless
Moving or speaking so slowly that other people could have noticed? Or the opposite – being so fidgety or restless that you have been moving around a lot more than usual


########################
Scale: gad_sum, Subset_id: 0
########################
Feeling nervous, anxious, or on edge
Not being able to stop or control worrying
Worrying too much about different things
Trouble relaxing
Being so restless that it is hard to sit still
Feeling afraid, as if something awful might happen
---------